# Guide intelligent du Musée

**Reconnaître le style d'une œuvre et répondre aux questions des visiteurs**

Projet final du cours *Neural Networks & Deep Learning Foundations*.

Membres de l'équipe : Bastien Ragueneau, Colin Pietri, Cristian Larrain.

Ce notebook est le rendu du projet : chaque chiffre cité est mesuré dans une cellule.

## 0. Contexte

Un musée des beaux-arts français veut un guide intelligent, sur le téléphone des visiteurs et sur des bornes dans les salles :

1. **Reconnaître une œuvre** : le visiteur photographie un tableau, le guide en reconnaît le style et raconte l'histoire du mouvement.
2. **Répondre en langage naturel** : à une question comme « Quel tableau montre une mer déchaînée ? », le guide propose les œuvres de la collection les plus proches.

La direction attend une recommandation claire : les bornes tournent sur des CPU modestes, l'équipe est petite et le budget limité. **Un réseau de neurones vaut-il le coup en production, ou un modèle classique suffit-il ?**

### Feuille de route

| Jalon | Contenu | Section | État |
|---|---|---|---|
| Séance 1 | Dépôt, choix de 5 styles, extraction du sous-ensemble WikiArt | 2 | fait |
| `m1` | Réseau dense de référence, entraîné avec une boucle propre | 3 | à faire |
| `m2` | Courbes de suivi train / validation et diagnostic des échecs | 4 | à faire |
| `m3` | ResNet pré-entraîné, comparé au réseau dense | 5 | à faire |
| `m4` | Recherche par question (embeddings de phrases) et comparaison neuronal / classique | 6 | à faire |

### Protocole

- Découpage train / validation / test décidé avant tout le reste, sans fuite : un même artiste n'apparaît que dans un seul split, et les statistiques de normalisation viennent du train seul.
- Graines fixées. Un gain faible n'est annoncé qu'avec la moyenne et l'écart sur 3 graines.
- Un modèle simple d'abord : il donne le chiffre à battre.
- Un seul changement à la fois, chaque run noté.
- Le test n'est utilisé qu'une fois, à la fin (section 7).

## 1. Mise en place

In [ ]:
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
SEED = 0
DATA_DIR = Path("data/wikiart_style5")  # relatif au dossier code/, où se trouve ce notebook

if next(DATA_DIR.glob("*/*.jpg"), None) is None:
    raise FileNotFoundError(
        f"Aucune image dans {DATA_DIR.resolve()} : générez le sous-ensemble avec la commande du README."
    )


def set_seed(seed: int = SEED) -> None:
    """Fixe les graines de Python, NumPy et PyTorch pour reproduire un run."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"PyTorch {torch.__version__}, calculs sur : {DEVICE}")

## 2. Données

Sous-ensemble de [`huggan/wikiart`](https://huggingface.co/datasets/huggan/wikiart) extrait par `data/make_wikiart_subset.py` : 5 styles contrastés, 300 œuvres par style, recadrées en 128 × 128 px. `metadata.csv` décrit chaque œuvre, `subset_info.json` garde la commande et la révision du dataset.

### 2.1 Chargement

In [ ]:
meta = pd.read_csv(DATA_DIR / "metadata.csv")
info = json.loads((DATA_DIR / "subset_info.json").read_text())

CLASSES = info["classes"]  # ordre alphabétique = label_idx = ordre d'ImageFolder
N_CLASSES = len(CLASSES)
assert list(meta.groupby("label_idx")["label"].first()) == CLASSES

print(f"{len(meta)} œuvres, {N_CLASSES} styles : {', '.join(CLASSES)}")
meta.head()

In [ ]:
def read_rgb(path: Path) -> np.ndarray:
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB"))


t0 = time.time()
X_all = torch.from_numpy(np.stack([read_rgb(DATA_DIR / p) for p in meta["path"]]))
X_all = X_all.permute(0, 3, 1, 2).contiguous()  # (N, H, W, C) -> (N, C, H, W), uint8
y_all = torch.tensor(meta["label_idx"].to_numpy())

print(f"X_all : {tuple(X_all.shape)} {X_all.dtype}, y_all : {tuple(y_all.shape)} {y_all.dtype}, "
      f"chargées en {time.time() - t0:.1f} s")

### 2.2 Exploration

Le script plafonne à 50 le nombre d'œuvres d'un même artiste nommé, mais pas celles de « Unknown Artist ».

In [ ]:
unknown = meta["artist"] == "Unknown Artist"
summary = meta.assign(inconnu=unknown).groupby("label").agg(
    oeuvres=("path", "size"),
    artistes_nommes=("artist", lambda a: a[a != "Unknown Artist"].nunique()),
    pct_artiste_inconnu=("inconnu", "mean"),
)
summary["pct_artiste_inconnu"] = (100 * summary["pct_artiste_inconnu"]).round(1)
summary

In [ ]:
# Genres représentés dans chaque style : un style peut se confondre avec un genre dominant.
pd.crosstab(meta["label"], meta["genre"])

In [ ]:
N_SHOW = 6
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(N_CLASSES, N_SHOW, figsize=(1.6 * N_SHOW, 1.6 * N_CLASSES))
for c, name in enumerate(CLASSES):
    for ax, i in zip(axes[c], rng.choice(np.flatnonzero(y_all == c), size=N_SHOW, replace=False)):
        ax.imshow(X_all[i].permute(1, 2, 0))
        ax.set_xticks([])
        ax.set_yticks([])
    axes[c, 0].set_ylabel(name, rotation=0, ha="right", va="center")
fig.suptitle("Six œuvres tirées au hasard par style")
plt.tight_layout()
plt.show()

### 2.3 Découpage train / validation / test

60 / 20 / 20, stratifié par style. Les œuvres d'un même artiste nommé restent dans un seul split : sinon le modèle pourrait reconnaître le peintre plutôt que le style. Chaque œuvre d'artiste inconnu forme son propre groupe.

In [ ]:
groups = np.where(unknown, "inconnu_" + meta["id"], meta["artist"])


def stratified_group_split(y: np.ndarray, groups: np.ndarray, n_splits: int, seed: int):
    """Met de côté un pli (environ 1/n_splits des œuvres), stratifié par style, sans couper de groupe."""
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return next(sgkf.split(np.zeros(len(y)), y, groups))


y_np = y_all.numpy()
rest, test_idx = stratified_group_split(y_np, groups, n_splits=5, seed=SEED)
train_pos, val_pos = stratified_group_split(y_np[rest], groups[rest], n_splits=4, seed=SEED)
train_idx, val_idx = rest[train_pos], rest[val_pos]
splits = {"train": train_idx, "val": val_idx, "test": test_idx}

# Contrôles : partition complète, aucun artiste nommé dans deux splits.
assert sorted(np.concatenate(list(splits.values()))) == list(range(len(meta)))
named_artists = {k: set(meta.loc[i, "artist"][~unknown[i]]) for k, i in splits.items()}
assert not (named_artists["train"] & named_artists["val"]
            or named_artists["train"] & named_artists["test"]
            or named_artists["val"] & named_artists["test"])

pd.DataFrame({k: meta.loc[i, "label"].value_counts() for k, i in splits.items()}).loc[CLASSES]

### 2.4 Statistiques de normalisation

Moyenne et écart-type par canal, calculés sur le train uniquement. Le ResNet utilisera à la place les statistiques d'ImageNet.

In [ ]:
X_train_float = X_all[train_idx].float() / 255
MEAN = X_train_float.mean(dim=(0, 2, 3))
STD = X_train_float.std(dim=(0, 2, 3))
print(f"moyenne (R, V, B) : {MEAN.numpy().round(3)}, écart-type : {STD.numpy().round(3)}")

## 3. Réseau dense de référence (jalon `m1`)

- Réseau feed-forward écrit avec les briques de PyTorch (`nn.Module`, `nn.Linear`, activations) sur les pixels aplatis : 3 × 128 × 128 = 49 152 entrées.
- Boucle d'entraînement propre : forward, perte, `zero_grad`, `backward`, `step` ; `eval()` et `no_grad()` pour évaluer.
- Vérifications : perte initiale proche de ln 5 ≈ 1,61, le réseau sait sur-apprendre un seul batch.
- Modèle classique à battre : régression logistique sur les mêmes pixels.
- Mesures : accuracy de validation, matrice de confusion, erreur la plus fréquente.

In [ ]:
# À compléter : jalon m1

## 4. Suivi de l'entraînement et diagnostic des échecs (jalon `m2`)

- Courbes de perte et d'accuracy, train et validation, à chaque époque.
- Diagnostic à partir des courbes : sur-apprentissage, taux d'apprentissage trop fort ou trop faible, réseau qui n'apprend rien.
- Corrections testées une par une : weight decay, dropout, arrêt précoce, batch norm, optimiseur.
- Analyse des erreurs : œuvres mal classées, styles confondus, rôle des artistes inconnus.

In [ ]:
# À compléter : jalon m2

## 5. ResNet pré-entraîné (jalon `m3`)

- ResNet-18 avec ses poids ImageNet, nouvelle tête à 5 sorties, normalisation ImageNet.
- Tête seule avec le backbone gelé, puis fine-tuning de `layer4` avec un taux d'apprentissage plus faible.
- Comparaison avec le réseau dense : accuracy, mais aussi temps d'inférence par image sur CPU et taille du modèle (contrainte des bornes).

In [ ]:
# À compléter : jalon m3

## 6. Questions en langage naturel (jalon `m4`)

- Une description textuelle par œuvre, construite à partir des métadonnées (style, genre, artiste).
- Embeddings de phrases (`all-MiniLM-L6-v2`) pour les questions et les descriptions, puis l'œuvre de plus forte similarité cosinus.
- Modèle classique de comparaison : TF-IDF.
- Jeu de questions de test avec les œuvres attendues, pour mesurer les deux approches.

In [ ]:
# À compléter : jalon m4

## 7. Évaluation finale et recommandation

- Évaluation unique sur le test des modèles retenus.
- Tableau comparatif : accuracy, temps d'inférence sur CPU, taille, coût d'entraînement.
- Recommandation à la direction : réseau de neurones ou modèle classique, pour chaque fonction du guide.

## 8. Limites

- **Artistes inconnus** : toutes les œuvres Impressionism et Pop_Art sont attribuées à « Unknown Artist » (section 2.2). Pour ces deux styles, le découpage ne peut pas garantir qu'un même peintre reste dans un seul split.
- **Genres peu renseignés** : 1 229 œuvres sur 1 500 (82 %) ont le genre « Unknown Genre » (section 2.2), ce qui appauvrit les descriptions textuelles du jalon `m4`.